# FixMatch Label-Fraction Sweep on STL-10

Trains FixMatch at label fractions of 4 / 10 / 25 / 100 / 500 per class with:
- **Random init** — standard FixMatch baseline
- **SimCLR init** — encoder from `simclr_pretrain.ipynb` (if checkpoint exists)

Produces a label efficiency curve and exports the best model to ONNX.

**Full run** (~90 min, Kaggle T4): `SMOKE_TEST = False`  
**Quick sanity check** (~3 min, CPU/low-disk): `SMOKE_TEST = True`

## 1. Configuration

In [ ]:
# ── Set to True for a fast local sanity-check (uses < 300 MB of data) ──────
SMOKE_TEST = False
# ────────────────────────────────────────────────────────────────────────────

DATA_DIR       = '/kaggle/working/data/'
CHECKPOINT_DIR = '/kaggle/working/checkpoints/'
PLOTS_DIR      = '/kaggle/working/plots/'
SIMCLR_CKPT    = f'{CHECKPOINT_DIR}simclr_encoder.pt'

# Smoke-test caps (ignored when SMOKE_TEST = False)
SMOKE_MAX_LABELLED   = 200
SMOKE_MAX_UNLABELLED = 500
SMOKE_EPOCHS_FM      = 2
SMOKE_EPOCHS_SUP     = 2
SMOKE_LABELS_SWEEP   = [25]

FULL_LABELS_SWEEP = [4, 10, 25, 100, 500]
LABELS_PER_CLASS  = SMOKE_LABELS_SWEEP if SMOKE_TEST else FULL_LABELS_SWEEP

print(f'SMOKE_TEST = {SMOKE_TEST}')
print(f'Label fractions: {LABELS_PER_CLASS}')

## 2. Environment setup

In [ ]:
import subprocess, sys
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'No GPU.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '--quiet'], check=True)
print('Package installed.')

In [ ]:
import json
from pathlib import Path

import mlflow
import torch
from torch.utils.data import DataLoader
from torchvision import datasets

from semi_supervised_image_clf.config import load_fixmatch_config, load_supervised_config
from semi_supervised_image_clf.dataset import get_stl10_splits
from semi_supervised_image_clf.evaluate import evaluate
from semi_supervised_image_clf.export import export_onnx
from semi_supervised_image_clf.fixmatch import FixMatchUnlabelledDataset, train_fixmatch
from semi_supervised_image_clf.model import ResNet18Classifier
from semi_supervised_image_clf.plot import plot_label_efficiency_curve
from semi_supervised_image_clf.supervised import train_supervised

for d in [DATA_DIR, CHECKPOINT_DIR, PLOTS_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)
mlflow.set_tracking_uri('file:///kaggle/working/mlruns')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}' + (f'  GPU: {torch.cuda.get_device_name(0)}' if DEVICE == 'cuda' else ''))
simclr_available = Path(SIMCLR_CKPT).exists()
print(f'SimCLR checkpoint available: {simclr_available}')

## 3. Load config

In [ ]:
fm_cfg  = load_fixmatch_config('config/fixmatch.yaml')
sup_cfg = load_supervised_config('config/supervised.yaml')

fm_cfg.data.data_dir  = DATA_DIR
sup_cfg.data.data_dir = DATA_DIR

if SMOKE_TEST:
    fm_cfg.training.epochs   = SMOKE_EPOCHS_FM
    sup_cfg.training.epochs  = SMOKE_EPOCHS_SUP
    fm_cfg.data.num_workers  = 2
    sup_cfg.data.num_workers = 2
else:
    fm_cfg.training.epochs   = 200
    sup_cfg.training.epochs  = 100
    fm_cfg.data.num_workers  = 4
    sup_cfg.data.num_workers = 4

INPUT_SIZE  = fm_cfg.model.input_size
NUM_CLASSES = fm_cfg.model.num_classes
SEED        = fm_cfg.data.random_seed
print(f'Input: {INPUT_SIZE}x{INPUT_SIZE}  Classes: {NUM_CLASSES}  Epochs: {fm_cfg.training.epochs}')

## 4. Download data

In [ ]:
# train (~120 MB) and test (~190 MB) are always needed.
# unlabelled (~2.5 GB) is only downloaded in full-run mode;
# in smoke mode it downloads once but only the first SMOKE_MAX_UNLABELLED images are used.
print('Downloading STL-10 train + test splits...')
for split in ('train', 'test'):
    ds = datasets.STL10(root=DATA_DIR, split=split, download=True)
    print(f'  {split}: {len(ds):,} images')

print('Downloading STL-10 unlabelled split...')
ds_unlab = datasets.STL10(root=DATA_DIR, split='unlabeled', download=True)
n_unlab = SMOKE_MAX_UNLABELLED if SMOKE_TEST else len(ds_unlab)
print(f'  unlabelled: using {n_unlab:,} of {len(ds_unlab):,} images')

## 5. Supervised baseline

In [ ]:
sup_cfg.data.labels_per_class = 500
labelled_loader, _, test_loader = get_stl10_splits(
    config=sup_cfg.data, labels_per_class=500, seed=SEED, input_size=INPUT_SIZE,
    smoke_test=SMOKE_TEST,
    max_labelled=SMOKE_MAX_LABELLED,
    max_unlabelled=SMOKE_MAX_UNLABELLED,
)
model_sup = train_supervised(
    ResNet18Classifier(num_classes=NUM_CLASSES),
    labelled_loader, test_loader, sup_cfg, CHECKPOINT_DIR,
)
acc_sup = evaluate(model_sup, test_loader).accuracy
print(f'Supervised baseline accuracy: {acc_sup:.4f}')

## 6. FixMatch label-fraction sweep

In [ ]:
results: dict[str, dict[int, float]] = {
    'Supervised':        {500 * NUM_CLASSES: acc_sup},
    'FixMatch':          {},
    'SimCLR + FixMatch': {},
}

for n in LABELS_PER_CLASS:
    total = n * NUM_CLASSES
    print(f'\n--- labels_per_class={n}  (total={total}) ---')

    fm_cfg.data.labels_per_class = n
    labelled_loader, unlabelled_base, test_loader = get_stl10_splits(
        config=fm_cfg.data, labels_per_class=n, seed=SEED, input_size=INPUT_SIZE,
        smoke_test=SMOKE_TEST,
        max_labelled=SMOKE_MAX_LABELLED,
        max_unlabelled=SMOKE_MAX_UNLABELLED,
    )
    unlabelled_loader = DataLoader(
        FixMatchUnlabelledDataset(unlabelled_base.dataset, input_size=INPUT_SIZE),
        batch_size=32 if SMOKE_TEST else fm_cfg.training.batch_size_unlabelled,
        shuffle=True,
        num_workers=fm_cfg.data.num_workers,
        pin_memory=(DEVICE == 'cuda'),
        drop_last=True,
    )

    # FixMatch — random init
    fm_cfg.model.pretrained_simclr = None
    model_fm = train_fixmatch(
        ResNet18Classifier(num_classes=NUM_CLASSES),
        labelled_loader, unlabelled_loader, fm_cfg, CHECKPOINT_DIR,
        run_name=f'fixmatch_n{n}',
    )
    results['FixMatch'][total] = evaluate(model_fm, test_loader).accuracy
    print(f'  FixMatch:          {results["FixMatch"][total]:.4f}')

    # SimCLR + FixMatch
    if simclr_available:
        model_ssl = ResNet18Classifier(num_classes=NUM_CLASSES)
        model_ssl.encoder.load_state_dict(torch.load(SIMCLR_CKPT, map_location='cpu'))
        fm_cfg.model.pretrained_simclr = SIMCLR_CKPT
        model_ssl = train_fixmatch(
            model_ssl, labelled_loader, unlabelled_loader, fm_cfg, CHECKPOINT_DIR,
            run_name=f'simclr+fixmatch_n{n}',
        )
        results['SimCLR + FixMatch'][total] = evaluate(model_ssl, test_loader).accuracy
        print(f'  SimCLR + FixMatch: {results["SimCLR + FixMatch"][total]:.4f}')

print('\nSweep complete.')

## 7. Results table

In [ ]:
with open('/kaggle/working/results.json', 'w') as f:
    json.dump(results, f, indent=2)

all_labels = sorted({k for d in results.values() for k in d})
methods = list(results.keys())
header = f"{'Labels':>8} | " + ' | '.join(f'{m:>22}' for m in methods)
print(header)
print('-' * len(header))
for n_labels in all_labels:
    row = f'{n_labels:>8} | '
    row += ' | '.join(f"{results[m].get(n_labels, float('nan')):>21.4f} " for m in methods)
    print(row)

## 8. Label efficiency curve

In [ ]:
from IPython.display import Image
save_path = f'{PLOTS_DIR}label_efficiency.png'
plot_label_efficiency_curve({k: v for k, v in results.items() if v}, save_path=save_path)
Image(save_path)

## 9. Export best model to ONNX

In [ ]:
best_acc, best_ckpt = 0.0, None
for ckpt in sorted(Path(CHECKPOINT_DIR).glob('*.pt')):
    try:
        m = ResNet18Classifier(num_classes=NUM_CLASSES)
        m.load_state_dict(torch.load(ckpt, map_location='cpu'))
        acc = evaluate(m, test_loader).accuracy
        print(f'  {ckpt.name}: {acc:.4f}')
        if acc > best_acc:
            best_acc, best_ckpt = acc, str(ckpt)
    except Exception as e:
        print(f'  {ckpt.name}: skip ({e})')

if best_ckpt:
    onnx_out = '/kaggle/working/best_model.onnx'
    export_onnx(best_ckpt, onnx_out, num_classes=NUM_CLASSES, input_size=INPUT_SIZE)
    print(f'Best: {Path(best_ckpt).name}  acc={best_acc:.4f}  -> {onnx_out}')